# Save qwen2.5:14b as a Kaggle Dataset (Ollama blobs)

Run this notebook **once** to permanently save the model as a Kaggle dataset.  
After that, every future notebook just mounts the dataset — no re-downloading.

### Before running
- Enable **GPU T4 x2** in Notebook Settings → Accelerator
- Enable **Internet** in Notebook Settings → Internet on
- Add your Kaggle API credentials as secrets (`KAGGLE_USERNAME` and `KAGGLE_KEY`)  
  → Get them from: https://www.kaggle.com/settings → API → Create New Token  
  → In Kaggle notebook: Add-ons → Secrets → add both

In [1]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
!nvidia-smi

Wed May  6 13:02:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ── Cell 2: Install Ollama ─────────────────────────────────────────────────
import os
os.environ["PATH"] += ":/usr/local/bin"

print("Installing zstd...")
!apt-get install -y zstd -q

print("Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh
print("Done.")

Installing zstd...
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (3,400 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
Installing Ollama...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%     83.4%
>>> Creating ollama user...
>>> Adding ollama user t

In [3]:
# ── Cell 3: Start Ollama server ────────────────────────────────────────────
import subprocess, time, os

# Add ollama to PATH for this session
os.environ["PATH"] += ":/usr/local/bin"

proc = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(5)
print("Ollama running, PID:", proc.pid)

Ollama running, PID: 206


In [4]:
# ── Cell 4: Pull the models ────────────────────────────────────────────────
# qwen2.5:14b  ~  8.9 GB download
# nomic-embed-text  ~  274 MB download

print("Pulling qwen2.5:14b — takes ~10 min...")
!ollama pull qwen2.5:14b

print("\nPulling nomic-embed-text...")
!ollama pull nomic-embed-text

print("\nAll models:")
!ollama list

Pulling qwen2.5:14b — takes ~10 min...
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 2049f5674b1e:   0% ▕                  ▏ 4.1 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   1% ▕                  ▏  77 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   2% ▕                  ▏ 165 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   2% ▕                  ▏ 217 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   3% ▕                  ▏ 309 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   4% ▕                  ▏ 401 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   5% ▕                  ▏ 451 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   6% ▕█                 ▏ 544 MB/9.0 GB                  pullin

In [9]:
# ── Cell 5: Verify where Ollama stored the blobs ──────────────────────────
import os

OLLAMA_MODELS_DIR = os.path.expanduser("~/.ollama/models")
print("Ollama models dir:", OLLAMA_MODELS_DIR)

for root, dirs, files in os.walk(OLLAMA_MODELS_DIR):
    level = root.replace(OLLAMA_MODELS_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:  # Don't print every blob filename
        subindent = '  ' * (level + 1)
        for file in files:
            size_mb = os.path.getsize(os.path.join(root, file)) / (1024**2)
            print(f"{subindent}{file}  ({size_mb:.1f} MB)")

Ollama models dir: /root/.ollama/models
models/
  manifests/
    registry.ollama.ai/
      library/
        nomic-embed-text/
        qwen2.5/
  blobs/
    sha256-970aa74c0a90ef7482477cf803618e776e173c007bf957f635f1015bfcfef0e6  (261.6 MB)
    sha256-832dd9e00a68dd83b3c3fb9f5588dad7dcf337a0db50f7d9483f310cd292e92e  (0.0 MB)
    sha256-eb4402837c7829a690fa845de4d7f3fd842c2adee476d5341da8a46ea9255175  (0.0 MB)
    sha256-ce4a164fc04605703b485251fe9f1a181688ba0eb6badb80cc6335c0de17ca0d  (0.0 MB)
    sha256-31df23ea7daa448f9ccdbbcecce6c14689c8552222b80defd3830707c0139d4f  (0.0 MB)
    sha256-2bada8a7450677000f678be90653b85d364de7db25eb5ea54136ada5f3933730  (4466.1 MB)
    sha256-c71d239df91726fc519c6eb72d318ec65820627232b2f796219e87dcf35d0ab4  (0.0 MB)
    sha256-66b9ea09bd5b7099cbb4fc820f31b575c0366fa439b08245566692c6784e281e  (0.0 MB)
    sha256-2f15b3218f0552c60647ce60ada83632d2c09755b16259b13e3e4458e9ae419d  (0.0 MB)


In [5]:
import shutil, os

SRC  = os.environ.get("OLLAMA_MODELS", os.path.expanduser("~/.ollama/models"))
DEST = "/kaggle/working/ollama-qwen25-14b"

print(f"Copying {SRC} → {DEST}")
shutil.copytree(SRC, DEST, dirs_exist_ok=True)

total = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, filenames in os.walk(DEST)
    for f in filenames
)
print(f"Copied {total / (1024**3):.2f} GB to {DEST}")

Copying /root/.ollama/models → /kaggle/working/ollama-qwen25-14b
Copied 8.63 GB to /kaggle/working/ollama-qwen25-14b


In [6]:
# ── Cell 7: Create dataset metadata ───────────────────────────────────────
import json
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
KAGGLE_USERNAME = secrets.get_secret("KAGGLE_USERNAME")

metadata = {
    "title": "ollama-qwen25-14b-nomic",
    "id": f"{KAGGLE_USERNAME}/ollama-qwen25-14b-nomic",
    "licenses": [{"name": "other"}]
}

with open("/kaggle/working/ollama-qwen25-14b/dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata written:")
print(json.dumps(metadata, indent=2))

Metadata written:
{
  "title": "ollama-qwen25-14b-nomic",
  "id": "ananyasrinivas/ollama-qwen25-14b-nomic",
  "licenses": [
    {
      "name": "other"
    }
  ]
}


In [7]:
# ── Cell 8: Set up Kaggle API credentials for CLI upload ──────────────────
import json, os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
KAGGLE_USERNAME = secrets.get_secret("KAGGLE_USERNAME")
KAGGLE_KEY      = secrets.get_secret("KAGGLE_KEY")

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print(f"Kaggle credentials set for user: {KAGGLE_USERNAME}")

Kaggle credentials set for user: ananyasrinivas


In [8]:
# # ── Cell 9: Upload dataset to Kaggle ──────────────────────────────────────
# # This zips and uploads — takes ~15-30 min depending on connection speed
# # The dataset will appear at:
# #   https://www.kaggle.com/datasets/YOUR_USERNAME/ollama-qwen25-14b-nomic

# print("Uploading to Kaggle... (this takes a while — ~9 GB)")
# !kaggle datasets create -p /kaggle/working/ollama-qwen25-32b-new --dir-mode zip
# print("\nUpload complete! ✅")

# ── Cell 9: Upload dataset to Kaggle ──────────────────────────────────────
import subprocess

# Upgrade kaggle CLI first (fixes the version warning)
!pip install kaggle --upgrade -q

print("Uploading to Kaggle... (this takes a while — ~9 GB for 14b)")

result = subprocess.run(
    [
        "kaggle", "datasets", "version",
        "-p", "/kaggle/working/ollama-qwen25-14b",
        "--dir-mode", "zip",
        "-m", "Qwen2.5 14b model upload"
    ],
    capture_output=True, text=True
)

print(result.stdout)

if result.returncode != 0:
    print("❌ Upload failed:")
    print(result.stderr)
else:
    print("Upload complete! ✅")
    print(f"View at: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/ollama-qwen25-14b")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 7.9 MB/s eta 0:00:00
Uploading to Kaggle... (this takes a while — ~9 GB for 14b)
Starting upload for file blobs.zip
Upload successful: blobs.zip (8GB)
Starting upload for file manifests.zip
Upload successful: manifests.zip (2KB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/ananyasrinivas/ollama-qwen25-14b-nomic

Upload complete! ✅
View at: https://www.kaggle.com/datasets/ananyasrinivas/ollama-qwen25-14b


## ✅ Done! What happens next

The dataset `YOUR_USERNAME/ollama-qwen25-14b-nomic` is now on Kaggle.

In your **MiroFish Kaggle notebook** (the one with ngrok):
1. Click **File → Add Input → Datasets**
2. Search for `ollama-qwen25-14b-nomic` (your own dataset)
3. Add it — it will be mounted at `/kaggle/input/ollama-qwen25-14b-nomic/`

Then replace the `ollama pull` cells with the fast restore cell below.